# HearAI — Phase 2 complète : CNN, YAMNet et AST sur ESC-50

Ce notebook regroupe **toute la Phase 2 dans un seul fichier** :
1. Construire, entraîner et évaluer le CNN.
2. Implémenter YAMNet et entraîner un classifieur sur ses embeddings.
3. Implémenter et fine-tuner AST.
4. Comparer les trois approches avec le même découpage officiel ESC-50.

**Découpage commun :** folds 1–3 pour train (1 200 fichiers), fold 4 pour validation (400), fold 5 pour test (400).

**Règle importante :** les choix de modèles se font sur la validation. Le test est utilisé à la fin pour l'évaluation finale, pas pour régler les hyperparamètres.

Le CNN conserve une configuration de régularisation de départ : dropout 0.5 + weight decay 0.01. Les améliorations supplémentaires pourront être étudiées plus tard.

## 0. Environnement Google Colab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install librosa soundfile tensorflow-hub transformers scikit-learn joblib


In [ ]:
import os, json, random, gc
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, f1_score
)
import joblib

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device PyTorch :', device)

BASE_DIR = Path('/content/drive/MyDrive/HearAI/sound')
RAW_DIR = BASE_DIR / 'data/raw/ESC-50'
AUDIO_DIR = RAW_DIR / 'audio'
META_PATH = RAW_DIR / 'meta/esc50.csv'
PROCESSED_DIR = BASE_DIR / 'data/processed'
MODELS_DIR = BASE_DIR / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

assert META_PATH.exists(), f'Métadonnées introuvables : {META_PATH}'
assert AUDIO_DIR.exists(), f'Dossier audio introuvable : {AUDIO_DIR}'


## 1. Charger ESC-50 et préparer le split commun

Nous repartons des fichiers audio bruts afin que les trois modèles reçoivent les données du même dataset et du même split. Le CNN convertira les audios en Mel-spectrogrammes ; YAMNet et AST recevront les formes audio attendues par leurs extracteurs.

In [ ]:
metadata = pd.read_csv(META_PATH)
metadata['fold'] = metadata['fold'].astype(int)

assert len(metadata) == 2000, f'Nombre de fichiers inattendu : {len(metadata)}'
assert metadata['category'].nunique() == 50, 'ESC-50 devrait contenir 50 classes.'

train_df = metadata[metadata.fold.isin([1, 2, 3])].reset_index(drop=True)
val_df   = metadata[metadata.fold == 4].reset_index(drop=True)
test_df  = metadata[metadata.fold == 5].reset_index(drop=True)

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_df['category'])
y_val   = label_encoder.transform(val_df['category'])
y_test  = label_encoder.transform(test_df['category'])
classes = label_encoder.classes_
NUM_CLASSES = len(classes)

print(f'Train      : {len(train_df)}')
print(f'Validation : {len(val_df)}')
print(f'Test       : {len(test_df)}')
print(f'Classes    : {NUM_CLASSES}')
assert (len(train_df), len(val_df), len(test_df)) == (1200, 400, 400)

with open(MODELS_DIR / 'esc50_classes.json', 'w', encoding='utf-8') as f:
    json.dump(classes.tolist(), f, ensure_ascii=False, indent=2)


## 2. Prétraitement audio commun

In [ ]:
SAMPLE_RATE = 16000
DURATION = 5
TARGET_LENGTH = SAMPLE_RATE * DURATION
N_MELS = 128
N_FFT = 2048
HOP_LENGTH = 512

def load_audio(filename):
    # Mono, rééchantillonné à 16 kHz et ramené à 5 secondes.
    path = AUDIO_DIR / filename
    waveform, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)

    if len(waveform) < TARGET_LENGTH:
        waveform = np.pad(waveform, (0, TARGET_LENGTH - len(waveform)))
    else:
        waveform = waveform[:TARGET_LENGTH]

    return waveform.astype(np.float32)

# Cache des formes d'onde pour éviter de relire les mêmes WAV pour chaque modèle.
audio_cache = {}
for i, filename in enumerate(metadata['filename'], start=1):
    audio_cache[filename] = load_audio(filename)
    if i % 250 == 0:
        print(f'{i}/2000 audios chargés')

print('Forme audio :', next(iter(audio_cache.values())).shape)


# Partie A — Modèle 1 : CNN

Le CNN travaille sur les Mel-spectrogrammes. On conserve ici une configuration proche de ton expérience actuelle : **Dropout = 0.5** et **Weight Decay = 0.01**. Ce n'est pas encore une recherche exhaustive d'hyperparamètres.

In [ ]:
def make_mel_spectrogram(waveform):
    mel = librosa.feature.melspectrogram(
        y=waveform,
        sr=SAMPLE_RATE,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        power=2.0
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = np.clip((mel_db + 80.0) / 80.0, 0.0, 1.0)
    return mel_db.astype(np.float32)

def build_mel_array(df):
    X = []
    for i, filename in enumerate(df['filename'], start=1):
        X.append(make_mel_spectrogram(audio_cache[filename]))
        if i % 250 == 0:
            print(f'{i}/{len(df)} Mel-spectrogrammes')
    return np.stack(X)

X_cnn_train = build_mel_array(train_df)
X_cnn_val = build_mel_array(val_df)
X_cnn_test = build_mel_array(test_df)

print('Train:', X_cnn_train.shape)
print('Validation:', X_cnn_val.shape)
print('Test:', X_cnn_test.shape)


In [ ]:
class MelDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X[:, None, :, :], dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH_SIZE_CNN = 32
train_cnn_loader = DataLoader(MelDataset(X_cnn_train, y_train),
                              batch_size=BATCH_SIZE_CNN, shuffle=True)
val_cnn_loader = DataLoader(MelDataset(X_cnn_val, y_val),
                            batch_size=BATCH_SIZE_CNN, shuffle=False)
test_cnn_loader = DataLoader(MelDataset(X_cnn_test, y_test),
                             batch_size=BATCH_SIZE_CNN, shuffle=False)


In [ ]:
class AudioCNN(nn.Module):
    def __init__(self, num_classes=50, dropout=0.5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.pool = nn.AdaptiveAvgPool2d((4, 4))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(64 * 4 * 4, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)

cnn_model = AudioCNN(NUM_CLASSES, dropout=0.5).to(device)
dummy = torch.randn(4, 1, 128, X_cnn_train.shape[-1]).to(device)
print('Sortie test CNN :', cnn_model(dummy).shape)


In [ ]:
criterion_cnn = nn.CrossEntropyLoss()
optimizer_cnn = torch.optim.Adam(
    cnn_model.parameters(),
    lr=0.001,
    weight_decay=0.01
)
EPOCHS_CNN = 30

def train_cnn_one_epoch(model, loader):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer_cnn.zero_grad()
        logits = model(inputs)
        loss = criterion_cnn(logits, labels)
        loss.backward()
        optimizer_cnn.step()

        n = labels.size(0)
        total_loss += loss.item() * n
        correct += (logits.argmax(1) == labels).sum().item()
        total += n
    return total_loss / total, correct / total

def evaluate_cnn(model, loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_targets, all_preds = [], []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            logits = model(inputs)
            loss = criterion_cnn(logits, labels)
            n = labels.size(0)
            total_loss += loss.item() * n
            correct += (logits.argmax(1) == labels).sum().item()
            total += n
            all_targets.extend(labels.cpu().numpy())
            all_preds.extend(logits.argmax(1).cpu().numpy())
    return total_loss / total, correct / total, np.array(all_targets), np.array(all_preds)

cnn_history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_cnn_val = -1.0
CNN_PATH = MODELS_DIR / 'cnn_dropout_05_weight_decay.pth'

for epoch in range(EPOCHS_CNN):
    tr_loss, tr_acc = train_cnn_one_epoch(cnn_model, train_cnn_loader)
    va_loss, va_acc, _, _ = evaluate_cnn(cnn_model, val_cnn_loader)

    cnn_history['train_loss'].append(tr_loss)
    cnn_history['train_acc'].append(tr_acc)
    cnn_history['val_loss'].append(va_loss)
    cnn_history['val_acc'].append(va_acc)

    if va_acc > best_cnn_val:
        best_cnn_val = va_acc
        torch.save(cnn_model.state_dict(), CNN_PATH)

    print(f'Epoch {epoch+1:02d}/{EPOCHS_CNN} | '
          f'Train loss {tr_loss:.4f} acc {tr_acc*100:.2f}% | '
          f'Val loss {va_loss:.4f} acc {va_acc*100:.2f}%')

print(f'Meilleure validation accuracy CNN : {best_cnn_val*100:.2f}%')
print('Checkpoint :', CNN_PATH)


In [ ]:
epochs = range(1, len(cnn_history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(epochs, cnn_history['train_loss'], label='Train')
axes[0].plot(epochs, cnn_history['val_loss'], label='Validation')
axes[0].set_title('CNN — Loss'); axes[0].set_xlabel('Époque'); axes[0].legend(); axes[0].grid(alpha=.3)
axes[1].plot(epochs, np.array(cnn_history['train_acc'])*100, label='Train')
axes[1].plot(epochs, np.array(cnn_history['val_acc'])*100, label='Validation')
axes[1].set_title('CNN — Accuracy (%)'); axes[1].set_xlabel('Époque'); axes[1].legend(); axes[1].grid(alpha=.3)
plt.tight_layout(); plt.show()


In [ ]:
# Recharger le meilleur CNN sélectionné sur la validation, puis évaluer le test une seule fois.
cnn_model.load_state_dict(torch.load(CNN_PATH, map_location=device))
cnn_test_loss, cnn_test_acc, cnn_true, cnn_pred = evaluate_cnn(cnn_model, test_cnn_loader)

print(f'CNN — Test accuracy : {cnn_test_acc*100:.2f}%')
print(classification_report(cnn_true, cnn_pred, labels=np.arange(NUM_CLASSES),
                            target_names=classes, zero_division=0))

cm = confusion_matrix(cnn_true, cnn_pred, labels=np.arange(NUM_CLASSES))
fig, ax = plt.subplots(figsize=(17, 15))
ConfusionMatrixDisplay(cm, display_labels=classes).plot(
    ax=ax, xticks_rotation=90, cmap='Blues', colorbar=False
)
ax.set_title('CNN — Matrice de confusion ESC-50 test')
plt.tight_layout(); plt.show()


# Partie B — Modèle 2 : YAMNet

YAMNet est pré-entraîné sur AudioSet. Ici, on extrait les embeddings YAMNet (vecteurs de 1024 dimensions), puis on entraîne une Logistic Regression sur les labels ESC-50. C'est un pipeline de transfert par extraction de caractéristiques, pas un fine-tuning complet de YAMNet.

In [ ]:
import tensorflow as tf
# Éviter que TensorFlow réserve toute la mémoire GPU avant l'utilisation de PyTorch/AST.
try:
    for gpu in tf.config.list_physical_devices('GPU'):
        tf.config.experimental.set_memory_growth(gpu, True)
except Exception as e:
    print('Configuration mémoire TensorFlow :', e)

import tensorflow_hub as hub
yamnet = hub.load('https://tfhub.dev/google/yamnet/1')
print('YAMNet chargé.')


In [ ]:
def get_yamnet_embedding(waveform):
    # YAMNet attend une forme d'onde mono float32 à 16 kHz.
    scores, embeddings, spectrogram = yamnet(
        tf.convert_to_tensor(waveform, dtype=tf.float32)
    )
    # Moyenne des embeddings sur les fenêtres temporelles.
    return tf.reduce_mean(embeddings, axis=0).numpy().astype(np.float32)

def extract_yamnet_features(df, split_name):
    features = []
    for i, filename in enumerate(df['filename'], start=1):
        features.append(get_yamnet_embedding(audio_cache[filename]))
        if i % 200 == 0:
            print(f'{split_name}: {i}/{len(df)}')
    return np.stack(features)

X_yam_train = extract_yamnet_features(train_df, 'Train')
X_yam_val = extract_yamnet_features(val_df, 'Validation')
X_yam_test = extract_yamnet_features(test_df, 'Test')

print(X_yam_train.shape, X_yam_val.shape, X_yam_test.shape)


In [ ]:
yamnet_classifier = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, C=1.0, random_state=SEED)
)
yamnet_classifier.fit(X_yam_train, y_train)

yam_val_pred = yamnet_classifier.predict(X_yam_val)
yam_val_acc = accuracy_score(y_val, yam_val_pred)
yam_val_f1 = f1_score(y_val, yam_val_pred, average='macro')
print(f'YAMNet — Validation accuracy : {yam_val_acc*100:.2f}%')
print(f'YAMNet — Validation macro F1 : {yam_val_f1:.4f}')


In [ ]:
yam_test_pred = yamnet_classifier.predict(X_yam_test)
yam_test_acc = accuracy_score(y_test, yam_test_pred)
print(f'YAMNet — Test accuracy : {yam_test_acc*100:.2f}%')
print(classification_report(y_test, yam_test_pred, labels=np.arange(NUM_CLASSES),
                            target_names=classes, zero_division=0))

cm_yam = confusion_matrix(y_test, yam_test_pred, labels=np.arange(NUM_CLASSES))
fig, ax = plt.subplots(figsize=(17, 15))
ConfusionMatrixDisplay(cm_yam, display_labels=classes).plot(
    ax=ax, xticks_rotation=90, cmap='Blues', colorbar=False
)
ax.set_title('YAMNet embeddings + Logistic Regression — ESC-50 test')
plt.tight_layout(); plt.show()

joblib.dump(yamnet_classifier, MODELS_DIR / 'yamnet_esc50_logistic.joblib')


# Partie C — Modèle 3 : AST

AST (Audio Spectrogram Transformer) est pré-entraîné sur AudioSet. Nous remplaçons sa tête de classification par une tête de 50 classes puis effectuons un fine-tuning sur le train ESC-50.

**Conseil :** activer un GPU Colab. AST est plus lourd que YAMNet. Si une erreur de mémoire apparaît, réduire `BATCH_SIZE_AST` à 4.

In [ ]:
from transformers import ASTFeatureExtractor, ASTForAudioClassification

AST_CHECKPOINT = 'MIT/ast-finetuned-audioset-10-10-0.4593'
ast_feature_extractor = ASTFeatureExtractor.from_pretrained(AST_CHECKPOINT)

id2label = {i: str(label) for i, label in enumerate(classes)}
label2id = {str(label): i for i, label in enumerate(classes)}

ast_model = ASTForAudioClassification.from_pretrained(
    AST_CHECKPOINT,
    num_labels=NUM_CLASSES,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
).to(device)

print('AST chargé sur :', device)


In [ ]:
class ASTAudioDataset(Dataset):
    def __init__(self, dataframe, labels):
        self.dataframe = dataframe.reset_index(drop=True)
        self.labels = np.asarray(labels, dtype=np.int64)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        filename = self.dataframe.loc[idx, 'filename']
        return audio_cache[filename], int(self.labels[idx])

def ast_collate_fn(batch):
    waveforms, labels = zip(*batch)
    encoded = ast_feature_extractor(
        list(waveforms),
        sampling_rate=SAMPLE_RATE,
        return_tensors='pt',
        padding=True
    )
    return encoded['input_values'], torch.tensor(labels, dtype=torch.long)

BATCH_SIZE_AST = 8
train_ast_loader = DataLoader(ASTAudioDataset(train_df, y_train),
                              batch_size=BATCH_SIZE_AST, shuffle=True,
                              collate_fn=ast_collate_fn, num_workers=0)
val_ast_loader = DataLoader(ASTAudioDataset(val_df, y_val),
                            batch_size=BATCH_SIZE_AST, shuffle=False,
                            collate_fn=ast_collate_fn, num_workers=0)
test_ast_loader = DataLoader(ASTAudioDataset(test_df, y_test),
                             batch_size=BATCH_SIZE_AST, shuffle=False,
                             collate_fn=ast_collate_fn, num_workers=0)

sample_x, sample_y = next(iter(train_ast_loader))
print('AST input batch :', sample_x.shape)


In [ ]:
def train_ast_epoch(model, loader, optimizer):
    model.train()
    loss_sum, count = 0.0, 0
    for input_values, labels in loader:
        input_values = input_values.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(input_values=input_values, labels=labels)
        outputs.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        n = labels.size(0)
        loss_sum += outputs.loss.item() * n
        count += n
    return loss_sum / count

def evaluate_ast(model, loader):
    model.eval()
    loss_sum, count = 0.0, 0
    all_targets, all_predictions = [], []
    with torch.no_grad():
        for input_values, labels in loader:
            input_values = input_values.to(device)
            labels = labels.to(device)
            outputs = model(input_values=input_values, labels=labels)

            n = labels.size(0)
            loss_sum += outputs.loss.item() * n
            count += n
            all_targets.extend(labels.cpu().numpy())
            all_predictions.extend(outputs.logits.argmax(dim=1).cpu().numpy())

    return (loss_sum / count,
            accuracy_score(all_targets, all_predictions),
            np.asarray(all_targets),
            np.asarray(all_predictions))


In [ ]:
EPOCHS_AST = 5
optimizer_ast = torch.optim.AdamW(
    ast_model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

ast_history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
best_ast_val = -1.0
AST_DIR = MODELS_DIR / 'ast_esc50'
AST_DIR.mkdir(parents=True, exist_ok=True)
AST_WEIGHTS = AST_DIR / 'best_model_state_dict.pth'

for epoch in range(EPOCHS_AST):
    train_loss = train_ast_epoch(ast_model, train_ast_loader, optimizer_ast)
    val_loss, val_acc, _, _ = evaluate_ast(ast_model, val_ast_loader)

    ast_history['train_loss'].append(train_loss)
    ast_history['val_loss'].append(val_loss)
    ast_history['val_acc'].append(val_acc)

    if val_acc > best_ast_val:
        best_ast_val = val_acc
        torch.save(ast_model.state_dict(), AST_WEIGHTS)
        ast_feature_extractor.save_pretrained(AST_DIR)

    print(f'Epoch {epoch+1}/{EPOCHS_AST} | '
          f'Train loss {train_loss:.4f} | '
          f'Val loss {val_loss:.4f} | '
          f'Val accuracy {val_acc*100:.2f}%')

print(f'Meilleure validation accuracy AST : {best_ast_val*100:.2f}%')
print('Checkpoint :', AST_WEIGHTS)


In [ ]:
epochs = range(1, len(ast_history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(epochs, ast_history['train_loss'], marker='o', label='Train')
axes[0].plot(epochs, ast_history['val_loss'], marker='o', label='Validation')
axes[0].set_title('AST — Loss'); axes[0].set_xlabel('Époque'); axes[0].legend(); axes[0].grid(alpha=.3)
axes[1].plot(epochs, np.array(ast_history['val_acc'])*100, marker='o', label='Validation')
axes[1].set_title('AST — Validation accuracy (%)'); axes[1].set_xlabel('Époque'); axes[1].legend(); axes[1].grid(alpha=.3)
plt.tight_layout(); plt.show()


In [ ]:
# Recharger le meilleur checkpoint choisi avec la validation.
ast_model.load_state_dict(torch.load(AST_WEIGHTS, map_location=device))
ast_test_loss, ast_test_acc, ast_true, ast_pred = evaluate_ast(ast_model, test_ast_loader)

print(f'AST — Test accuracy : {ast_test_acc*100:.2f}%')
print(classification_report(ast_true, ast_pred, labels=np.arange(NUM_CLASSES),
                            target_names=classes, zero_division=0))

cm_ast = confusion_matrix(ast_true, ast_pred, labels=np.arange(NUM_CLASSES))
fig, ax = plt.subplots(figsize=(17, 15))
ConfusionMatrixDisplay(cm_ast, display_labels=classes).plot(
    ax=ax, xticks_rotation=90, cmap='Blues', colorbar=False
)
ax.set_title('AST fine-tuné — ESC-50 test')
plt.tight_layout(); plt.show()

torch.save(ast_model.state_dict(), AST_WEIGHTS)
ast_feature_extractor.save_pretrained(AST_DIR)


# Partie D — Comparaison finale des trois modèles

Le tableau affiche les scores de validation et de test. Pour sélectionner un modèle destiné à la suite du projet, examine d'abord la validation et les métriques par classe. Le score test sert à rapporter la performance finale sur le fold réservé.

In [ ]:
comparison = pd.DataFrame([
    {
        'Modèle': 'CNN (Dropout 0.5 + Weight Decay 0.01)',
        'Validation accuracy (%)': round(best_cnn_val * 100, 2),
        'Test accuracy (%)': round(cnn_test_acc * 100, 2),
        'Approche': 'CNN entraîné depuis zéro sur Mel-spectrogrammes'
    },
    {
        'Modèle': 'YAMNet + Logistic Regression',
        'Validation accuracy (%)': round(yam_val_acc * 100, 2),
        'Test accuracy (%)': round(yam_test_acc * 100, 2),
        'Approche': 'Embeddings pré-entraînés + classifieur'
    },
    {
        'Modèle': 'AST fine-tuné',
        'Validation accuracy (%)': round(best_ast_val * 100, 2),
        'Test accuracy (%)': round(ast_test_acc * 100, 2),
        'Approche': 'Transformer pré-entraîné puis fine-tuné'
    }
])

display(comparison.sort_values('Validation accuracy (%)', ascending=False))
comparison.to_csv(MODELS_DIR / 'phase2_three_models_comparison.csv', index=False)
print('Comparaison sauvegardée :', MODELS_DIR / 'phase2_three_models_comparison.csv')


In [ ]:
# Comparaison visuelle des accuracies
plot_df = comparison.set_index('Modèle')[['Validation accuracy (%)', 'Test accuracy (%)']]
ax = plot_df.plot(kind='bar', figsize=(12, 5))
ax.set_ylabel('Accuracy (%)')
ax.set_title('HearAI — Comparaison CNN, YAMNet et AST sur ESC-50')
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()


## 16. Bilan de la Phase 2

À la fin de ce notebook, tu disposeras de :
- un CNN entraîné et évalué ;
- un pipeline YAMNet + Logistic Regression entraîné et évalué ;
- un AST fine-tuné et évalué ;
- les rapports de classification et matrices de confusion ;
- un tableau de comparaison sauvegardé dans `models/phase2_three_models_comparison.csv`.

### Points importants
- Les trois modèles utilisent les mêmes 1 200 exemples train, 400 validation et 400 test.
- YAMNet n'est pas fine-tuné dans cette version : ses embeddings sont extraits puis un classifieur est appris sur ESC-50.
- AST est fine-tuné sur ESC-50. Le temps d'entraînement dépend du GPU Colab.
- ESC-50 est un dataset de sons environnementaux généraux. Pour HearAI Safety Agent, il faudra ensuite mesurer en priorité le recall des événements critiques et les faux négatifs, avec des données représentatives des situations réelles.
- Les résultats ne sont pas préremplis : ils seront produits lors de l'exécution du notebook dans ton environnement Colab.

### Si AST manque de mémoire
1. Arrêter les autres sessions GPU Colab.
2. Redémarrer l'environnement si nécessaire.
3. Mettre `BATCH_SIZE_AST = 4` puis recréer les trois DataLoaders AST.
4. Si nécessaire, réduire `EPOCHS_AST` pour un test de fonctionnement, puis lancer l'entraînement complet ensuite.